<a href="https://colab.research.google.com/github/tqan27/DiscreteAssignment/blob/main/Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1 a. #779 Prime Factor and Exponent

In [ ]:
import time, math
start_time = time.time()

def min_prime_factor(n):
    result = [0]*(n+1)
    result[0] = [0,0]
    result[1] = [0,0]
    for x in range(2, n + 1):
        if result[x] == 0:
            for y in range(x, n + 1, x):
                if result[y] == 0:

                    power = 1
                    while y % x**(power +1) == 0:
                        power += 1

                    result[y] = [x, power]
    return result

def estimate(K, number):
    values = min_prime_factor(number)
    total = 0

    for x in range(2, number):
        smallest_prime, p_adic_order = values[x]
        total += (p_adic_order - 1)/(smallest_prime**K)

    return total/number

def list_primality(n):
	result = [True] * (n + 1)
	result[0] = result[1] = False
	for i in range(int(math.sqrt(n)) + 1):
		if result[i]:
			for j in range(2 * i, len(result), i):
				result[j] = False
	return result

def list_primes(n):
	return [i for (i, isprime) in enumerate(list_primality(n)) if isprime]

def f(K):
    primes = list_primes(10**6)
    total = 0
    curr = 1
    for p in primes:
        total += curr*(1/((p**(K+1))*(p-1)))
        curr *= (p - 1)/p
    return round(total, 12)

def sum_f():
    primes = list_primes(10**6)
    total = 0
    curr = 1
    for p in primes:
        total += curr*(1/((p*(p-1)**2)))
        curr *= (p - 1)/p
    return round(total, 12)

if __name__ == "__main__":
    print(sum_f())

0.547326103833


Q1 b. #152 Sums of Square Reciprocals

In [ ]:
from itertools import filterfalse # changed from ifilter to filterfalse

def isPrime(n):
    d = 2
    while d**2 <= n:
        if n % d == 0: return False
        d += 1
    return True

def subsetsGen(setOr):
    for s in subsetsGenAux(list(setOr), set([]), 0):
        yield s

def subsetsGenAux(xs, partial, n):
    if n >= len(xs):
        yield partial
    else:
        for s in subsetsGenAux(xs, partial, n+1): yield s
        partial.add(xs[n])
        for s in subsetsGenAux(xs, partial, n+1): yield s
        partial.remove(xs[n])

def factorsInList(n, ps):
    for p in ps:
        while n % p == 0: n //= p
    return n == 1

def mcm(ns):
    output = 1
    for el in ns: output = mcm2(output, el)
    return output

def mcm2(a, b):
    return (a * b) // gcd2(a, b)

def gcd2(a, b):
    if a > b: return gcd2aux(a, b)
    return gcd2aux(b, a)

def gcd2aux(a, b):
    while b: (a, b) = (b, a%b)
    return a

def polyProd(p1, p2, max=-1):
    '''Returns the product of the polynomials p1 and p2, except
    for those terms with degree greater that max.'''
    p = {}
    for (deg1, coef1) in p1.items():
        for (deg2, coef2) in p2.items():
            newdef = deg1 + deg2
            if max<0 or newdef <= max:
                p[newdef] = p.get(newdef, 0) + coef1*coef2
    return p

def polySet(s, max):
    p = {0:1}
    for el in s: p = polyProd(p, {0:1, el:1}, max)
    return p

def isValid(numer, denom, p):
    '''Returns whether the numerator has at least two more ps than
    the denominator, in the prime factor decompositions.'''
    while numer%p == 0 and denom%p == 0:
        numer //= p
        denom //= p
    return numer % (p**2) == 0


def possibleP(p):
    '''Generates the sets I such that p * I could be part of a solution.

    For example, for p = 7 we get [], [1,4,5], etc. This means that
    (7, 28, 35) could be part of a solution.'''
    for s in subsetsGen(range(1, limit//p + 1)):
        sumNs = sum(val[i] for i in s)
        if isValid(sumNs, mcmTot, p):
            yield s # Added indentation


## ****************** MAIN ******************

limit = 80
cands = range(2, limit + 1)
# Changed to list for Python 3 compatibility
cands = list(cands)
denoms = [x**2 for x in cands]
mcmTot = mcm(denoms)

## Global variable, used everywhere. The cases 0 and 1 are manually set.
val = [None, mcmTot] + [mcmTot // d for d in denoms]

## Removing candidates that don't appear in any combination:
sRemove = set([])
# Changed ifilter and xrange to filter and range for Python 3 compatibility
for p in filter(isPrime, range(5, limit + 1)):
    sFound = set(range(p, limit + 1, p))
    for s in possibleP(p):
        # Changed to a generator expression for Python 3 compatibility
        sFound.difference_update(p*el for el in s)
    sRemove.update(sFound)
for el in list(sRemove):
    if el in cands:
        cands.remove(el)

print("Candidates: %s" % cands)
## Now, we divide the candidates in approx. 2 halves:
denoms1 = [x**2 for x in cands[:len(cands)//2]]
denoms2 = [x**2 for x in cands[len(cands)//2:]]
mcm_1 = mcm(denoms1)
mcm_2 = mcm(denoms2)
mcmTot = mcm([mcm_1, mcm_2])
mcmTot2 = mcmTot // 2
s1 = [mcmTot // d for d in denoms1]
s2 = [mcmTot // d for d in denoms2]
p1 = polySet(s1, mcmTot2)
p2 = polySet(s2, mcmTot2)
print("Sums in part 1: %s" % len(p1))
print("Sums in part 2: %s" % len(p2))

counter = 0
# Changed iteritems() to items() for Python 3 compatibility
for (g1, coef1) in p1.items():
    counter += coef1 * p2.get(mcmTot2 - g1, 0)
print("Number of solutions: %s" % counter)

Candidates: [2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 18, 20, 21, 24, 27, 28, 30, 32, 35, 36, 39, 40, 42, 45, 48, 52, 54, 56, 60, 63, 64, 70, 72, 80]
Sums in part 1: 354917
Sums in part 2: 371968
Number of solutions: 301


Q1 c. #179 Consecutive Positive Divisors

In [ ]:
def compute():
	divisors = [2] * (10**7 + 1)  # Invalid for indexes 0 and 1
	for i in range(2, (len(divisors) + 1) // 2):
		for j in range(i * 2, len(divisors), i):
			divisors[j] += 1

	ans = sum((1 if divisors[i] == divisors[i + 1] else 0) for i in range(2, len(divisors) - 1))
	return str(ans)


if __name__ == "__main__":
	print(compute())

986262


Q1 d. #763 Amoebas in a 3D Grid

---



In [ ]:
def p763():

    md = 10**9

    def dic_add(d, k, v):
        if k in d: d[k] = (d[k] + v) % md
        else: d[k] = v % md
        return

    def transitions(s):
        if s == ():
            yield (), 3
            yield (1,), 3
        elif s == (1,):
            yield (), 1
            yield (1,), 4
            yield (2,), 1
            yield (1, 1), 2
        elif len(s) == 1:
            x = s[0]
            yield (x-1,), 1
            yield (x,), 2
            yield (x+1,), 1
            yield (x, 1), 2
            yield (x-1, 1), 2
        else:
            x, y = s
            yield (x, y), 1
            yield (x, y+1), 1
            yield (x+y, 1), 1
            yield (x+y-1, 1), 1
            if x == 1:
                yield (y,), 1
                yield (y+1,), 1
            else:
                yield (x-1, y), 1
                yield (x-1, y+1), 1
        return

    def D(N):
        S = [{} for _ in range(N+1)]
        S[0][()] = 1
        for i in range(N-1):
            for s in S[i]:
                v = S[i][s]
                for t, w in transitions(s):
                    d = sum(t)+1
                    if i + d*(d+1)//2 > N:
                        continue
                    dic_add(S[i+d], t, v*w)
        return S[N-1][()]

    print(D(10000))

p763()

798443574
